In [55]:
import json
import requests
import base64
from bs4 import BeautifulSoup

def compute_correct_letter():
    # A string "Njk=" em Base64 decodifica para "69". 
    # A função bc2 subtrai 2 do número, ou seja, 69 - 2 = 67, que corresponde ao caractere "C".
    s = "Njk="
    decoded = base64.b64decode(s).decode('utf-8')
    number = int(decoded)
    return chr(number - 2)  # retorna "C"

def compute_correct_letter2():
    # A string "Njk=" em Base64 decodifica para "69". 
    # A função bc2 subtrai 2 do número, ou seja, 69 - 2 = 67, que corresponde ao caractere "C".
    s = "Njk="
    decoded = base64.b64decode(s).decode('utf-8')
    number = int(decoded)
    return chr(number - 3)  # retorna "C"

def compute_correct_letter3():
    # A string "Njk=" em Base64 decodifica para "69". 
    # A função bc2 subtrai 2 do número, ou seja, 69 - 2 = 67, que corresponde ao caractere "C".
    s = "Njk="
    decoded = base64.b64decode(s).decode('utf-8')
    number = int(decoded)
    return chr(number - 1)  # retorna "A"



def criar_corpus_bomcondutor(inicio=1, fim=10000):
    corpus = {"perguntas_respostas": []}
    headers = {'User-Agent': 'Mozilla/5.0'}

    correct_letter = compute_correct_letter()  # "C"
    correct_letter2 = compute_correct_letter2()  # "B"
    correct_letter3 = compute_correct_letter3()  # "A"

    for i in range(inicio, fim + 1):
        url = f"https://www.bomcondutor.pt/questao/{i}"
        print(f"Extraindo conteúdo de: {url}")

        resposta = requests.get(url, headers=headers)
        if resposta.status_code != 200:
            print(f"Erro ao acessar a questão {i}: Status code {resposta.status_code}")
            continue

        resposta.encoding = 'utf-8'
        soup = BeautifulSoup(resposta.text, 'html.parser')

        # Extraindo a pergunta
        pergunta_element = soup.select_one('.question-text .text')
        if not pergunta_element:
            print(f"Pergunta não encontrada na questão {i}")
            continue
        pergunta = pergunta_element.get_text(strip=True)
        print(f"Pergunta: {pergunta}")

        # Tenta extrair a resposta correta (caso o JS já tenha inserido a classe "correct")
        # Se não encontrou, assume que a resposta correta é a opção resultante do JS ("C")
        fallback_selector = f'li.answer.{correct_letter3} span.answer-text'
        resposta_element = soup.select_one(fallback_selector)
        if resposta_element:
            resposta_correta = resposta_element.get_text(strip=True)
            print(f"Resposta correta: ", correct_letter3)
        else:
            # Se não encontrou, assume que a resposta correta é a opção resultante do JS ("C")
            fallback_selector = f'li.answer.{correct_letter} span.answer-text'
            resposta_element = soup.select_one(fallback_selector)
            if resposta_element:
                resposta_correta = resposta_element.get_text(strip=True)
                print(f"Resposta correta: ", correct_letter)
            else:
                # Se não encontrou, assume que a resposta correta é a opção resultante do JS ("C")
                fallback_selector = f'li.answer.{correct_letter2} span.answer-text'
                resposta_element = soup.select_one(fallback_selector)
                if resposta_element:
                    resposta_correta = resposta_element.get_text(strip=True)
                    print(f"Resposta correta: ", correct_letter2)
                else:
                    print(f"Resposta correta não encontrada na questão {i}")
                    continue

        print(f"Resposta correta: {resposta_correta}")

        # Extraindo a explicação, se disponível
        explicacao_element = soup.select_one('div#explicacao .contents')
        explicacao = ""
        if explicacao_element:
            explicacao_texto = explicacao_element.get_text(strip=True)
            if "Esta questão ainda não possui conteúdo auxiliar." not in explicacao_texto:
                explicacao = explicacao_texto

        corpus_entry = {
            "padrao": pergunta,
            "respostas": [resposta_correta]
        }
        if explicacao:
            corpus_entry["explicacao"] = explicacao

        print("-" * 50)
        corpus["perguntas_respostas"].append(corpus_entry)

    with open("corpus_bomcondutor.json", "w", encoding="utf-8") as arquivo:
        json.dump(corpus, arquivo, ensure_ascii=False, indent=4)

    print(f"Corpus criado com sucesso! Total de {len(corpus['perguntas_respostas'])} perguntas adicionadas.")

if __name__ == "__main__":
    criar_corpus_bomcondutor(1, 10)


Extraindo conteúdo de: https://www.bomcondutor.pt/questao/1
Pergunta: A distância de segurança deve ser sempre a que me permita imobilizar o ciclomotor sem perigo de acidente.
Resposta correta:  B
Resposta correta: Errado.
--------------------------------------------------
Extraindo conteúdo de: https://www.bomcondutor.pt/questao/2
Pergunta: A distância que devo guardar do ciclomotor depende:
Resposta correta:  C
Resposta correta: Da velocidade a que esse ciclomotor circula.
--------------------------------------------------
Extraindo conteúdo de: https://www.bomcondutor.pt/questao/3
Pergunta: A distância que devo guardar do ciclomotor que me precede deve:
Resposta correta:  C
Resposta correta: Ter em conta a velocidade a que esse ciclomotor circula.
--------------------------------------------------
Extraindo conteúdo de: https://www.bomcondutor.pt/questao/4
Pergunta: A pressão dos pneus deve ser a indicada pelo construtor do veículo.
Resposta correta:  B
Resposta correta: Errado.
---